# Case Study 2 - Question 1

Studio Delta got 30 candidate building designs from the engineering team, each scored on 4 criteria
(PV, daylight, compactness, FSI, all higher-is-better). We have to pick which one(s) to recommend.

Q1: analyse the design space and identify the most appropriate solution(s).

In [1]:
import numpy as np
import pandas as pd
np.random.seed(0)

configs    = pd.read_csv("../data/configurations.csv")
thresholds = pd.read_csv("../data/thresholds.csv")
CRIT = ["pv", "daylight", "compactness", "fsi"]
rows = configs.set_index("id")

In [2]:
# stakeholder minimums (Meeting 4)
mins = dict(zip(thresholds.criterion, thresholds.minimum))
mins

{'pv': 0.7, 'daylight': 0.7, 'compactness': 0.75, 'fsi': 0.8}

### Feasibility


In [4]:
feasible = [x for x in rows.index if all(rows.loc[x][c] >= mins[c] for c in CRIT)]
print(len(feasible), "feasible:", feasible)

# which threshold removes how many
for c in CRIT:
    n = sum(rows.loc[x][c] < mins[c] for x in rows.index)
    print(f"  {c:12s} >= {mins[c]}  removes {n}")

10 feasible: ['C1', 'C6', 'C8', 'C11', 'C14', 'C15', 'C18', 'C21', 'C25', 'C28']
  pv           >= 0.7  removes 10
  daylight     >= 0.7  removes 9
  compactness  >= 0.75  removes 8
  fsi          >= 0.8  removes 10


In [5]:
def shade(col):
    thr = mins[col.name]
    return ['background-color:#c6efce' if v >= thr else 'background-color:#ffc7ce' for v in col]

def label(idx):
    return ['font-weight:bold' if i in feasible else 'color:#aaa' for i in idx]

(rows[CRIT].style
    .apply(shade, axis=0)
    .apply_index(label, axis=0)
    .format('{:.2f}')
    .set_caption('green = passes threshold, red = fails; all-green rows are feasible'))

,pv,daylight,compactness,fsi
id,,,,
C1,0.80,0.75,0.81,0.88
C2,0.72,0.81,0.91,0.78
C3,0.86,0.63,0.74,0.96
C4,0.68,0.88,0.84,0.82
C5,0.91,0.58,0.69,0.98
C6,0.75,0.77,0.88,0.87
C7,0.62,0.92,0.79,0.72
C8,0.83,0.70,0.76,0.94
C9,0.57,0.95,0.86,0.66


### Choosing among the 10

Simplest idea is a weighted sum (weights x scores, add up). But nobody gave us weights, and the
winner depends on them.

In [6]:
profiles = {
    "equal":        [.25, .25, .25, .25],
    "FSI-heavy":    [.10, .10, .10, .70],
    "PV+daylight":  [.35, .35, .15, .15],
    "compactness":  [.15, .15, .55, .15],
}
for name, w in profiles.items():
    s = {x: float(np.dot(w, rows.loc[x, CRIT])) for x in feasible}
    print(f"{name:12s} -> {max(s, key=s.get)}")

equal        -> C6
FSI-heavy    -> C8
PV+daylight  -> C1
compactness  -> C14


Different weights, different winners. Since we don't have weights, try all of them: draw random
weights many times and count how often each design wins (this is SMAA).

In [7]:
Yf = rows.loc[feasible, CRIT].values
wins = {x: 0 for x in feasible}
N = 50000
for _ in range(N):
    w = np.random.dirichlet([1, 1, 1, 1])
    wins[feasible[np.argmax(Yf @ w)]] += 1

for x in sorted(feasible, key=lambda k: -wins[k]):
    print(f"{x:5s} {wins[x]/N*100:5.1f}%")

C8     38.3%
C14    30.4%
C6     20.1%
C21     7.4%
C1      2.9%
C18     0.9%
C11     0.0%
C15     0.0%
C25     0.0%
C28     0.0%


C6, C8 and C14 take almost all the wins. Three designs (C15, C25, C28) win 0% - not rarely,
never. No weighted sum can ever pick them, because each is beaten by a mix of the others.

In [8]:
# C15 vs a blend of C6, C8, C14, C21
blend_w = {"C6": .027, "C8": .782, "C14": .091, "C21": .100}
blend = sum(w * rows.loc[x, CRIT].values for x, w in blend_w.items())
for i, c in enumerate(CRIT):
    print(f"{c:12s} blend {blend[i]:.3f}   C15 {rows.loc['C15', c]:.2f}")

pv           blend 0.810   C15 0.81
daylight     blend 0.720   C15 0.72
compactness  blend 0.780   C15 0.78
fsi          blend 0.921   C15 0.92


The blend matches C15 on three criteria and beats it on FSI, so C15 can never be strictly best.
A weighted sum (and TOPSIS, additive AHP) misses 3 of the 10 valid designs whatever the weights, so
method choice matters here.

### Q1 answer

Recommend a set, not a single winner - the 1.5% spread makes one "best" design false precision.

- C6 - balanced, best average across all weightings
- C8 - strong on density and solar
- C14 - strong on compactness and daylight

These span the trade-off. Which one gets built depends on whose priorities count, which is Q2.

In [9]:
rows.loc[["C6", "C8", "C14"], CRIT].round(2)

,pv,daylight,compactness,fsi
id,,,,
C6,0.75,0.77,0.88,0.87
C8,0.83,0.70,0.76,0.94
C14,0.70,0.80,0.90,0.84
